# Neural Fortran Examples: Dense Network on MNIST

Created with Grok Build

Source code by [Milan Curcic](https://github.com/milancurcic) and the neural-fortran contributors, from the [neural-fortran](https://github.com/modern-fortran/neural-fortran) repository. The library is released under the MIT license. Copyright (c) 2018-2025 neural-fortran contributors. The paper is Milan Curcic, [A parallel Fortran framework for neural networks and deep learning](https://doi.org/10.1145/3323057.3323059), *ACM SIGPLAN Fortran Forum* 38(1), 2019.

## Introduction

This is `example/dense_mnist.f90`. It loads the library's MNIST files, trains a dense classifier for 10 epochs, and prints validation accuracy, a quadratic loss, and a Pearson correlation.

## Data and the network

```{literalinclude} Fortran_Code/Section_Examples_Dense_Mnist/app/dense_mnist.f90
:language: fortran
:lines: 1-29
```

`load_mnist` reads six binary files. If `mnist_training_images.dat` is missing, the library downloads `mnist.tar.gz` from the neural-fortran file store and unpacks it. This run found the files already unpacked in the working directory. The counts in the loader are 50000 training images, 10000 validation images, and 10000 test images. This program does not request the test arrays.

Each image is 784 floats, one per pixel, stored as a column. `label_digits` turns a digit $d$ into a length-10 vector that is zero except for a one in position $d+1$. Digit 0 is the first component.

The network is `input(784)`, `dense(64, relu())`, `dropout(0.2)`, and `dense(10, softmax())`. ReLU is $\max(z, 0)$. Softmax turns the 10 logits $z$ into

$$
\hat y_k = \frac{e^{z_k}}{\sum_{j=1}^{10} e^{z_j}}.
$$

The dense parameter counts are $784 \cdot 64 + 64 = 50240$ and $64 \cdot 10 + 10 = 650$. Dropout has none. The printed dropout rate is 0.20, and `predict` switches dropout to inference before it scores the validation set, then switches training mode back on.


## Training and the accuracy function

```{literalinclude} Fortran_Code/Section_Examples_Dense_Mnist/app/dense_mnist.f90
:language: fortran
:lines: 31-67
```

Each epoch calls `train` for one epoch with batch size 128 and SGD at learning rate 3. Inside `train`, an epoch is `50000 / 128 = 390` steps. Each step draws a random contiguous window of 128 images. Windows can overlap, and an image can be skipped. The window is not a shuffled partition of the dataset. `update` divides the accumulated gradient by the batch size before the SGD step, so the learning rate multiplies the mean gradient of the window.

`accuracy` counts a validation image as correct when `maxloc` of the 10 network outputs equals `maxloc` of the one-hot label. The denominator is 10000.

`evaluate` returns two columns. The first is the quadratic loss $L = \sum_k (\hat y_k - y_k)^2 / 2$. The second is the Pearson correlation

$$
r = \frac{(y-\bar y)\cdot(\hat y-\bar{\hat y})}{\sqrt{\|y-\bar y\|^2\ \|\hat y-\bar{\hat y}\|^2}}.
$$

The reduction in the program is `sum(output_metrics, 1) / size(output_metrics, 1)`. `predict_batch` stores outputs as `(10, n_images)`, and `evaluate` loops the first dimension. The mean is therefore the mean of 10 class-channel scores, and each channel score is computed across all 10000 validation images. The printed loss is not a per-image average. A per-image quadratic loss on a softmax would sit below 1. The values near 20 are sums over the validation set, averaged across the ten digit channels. The Pearson number is the same kind of average, which is why it can sit near 0.97 next to a loss near 22.


## Program

```{literalinclude} Fortran_Code/Section_Examples_Dense_Mnist/app/dense_mnist.f90
:language: fortran
```

The package file below builds this source with fpm. The Fortran program itself is the file from the repository.

```{literalinclude} Fortran_Code/Section_Examples_Dense_Mnist/fpm.toml
:language: toml
```

## Build

The program above was compiled from the neural-fortran repository at commit `357e63d` (`Support of the argument stride for locally_connected2d_layer`) with fpm 0.10.1:

```text
fpm build --profile release
```

The compiler was gfortran 14.2.0. The log is that release build. It compiles the library and every program in `example/`, including this one. The `fpm.toml` on this page is a standalone package for the copied source. It takes neural-fortran as a git dependency, in the same form as the sine-wave page.


```{literalinclude} Fortran_Code/fpm_release_build.log
:language: text
```

## Run

The binary was `dense_mnist` from the release build, run in a directory that already held the six MNIST files. Standard error was empty and the process exited 0. Accuracy used the 10000 validation images.

```{literalinclude} Fortran_Code/Section_Examples_Dense_Mnist/dense_mnist.out
:language: text
```

## What the run did

![Dense MNIST accuracy and loss](images/nf_dense_mnist.png)

Before any update the validation accuracy is 5.99 percent, which is 599 of 10000 images. An untrained network does not have to land near the 10 percent chance line.

| Epoch | Accuracy (%) | Printed loss | Pearson |
| ---: | ---: | ---: | ---: |
| 1 | 93.730 | 47.561 | 0.946 |
| 2 | 95.210 | 37.546 | 0.957 |
| 3 | 95.530 | 34.491 | 0.961 |
| 4 | 95.960 | 31.249 | 0.964 |
| 5 | 96.250 | 28.894 | 0.967 |
| 6 | 96.680 | 25.048 | 0.972 |
| 7 | 96.690 | 25.196 | 0.972 |
| 8 | 96.890 | 23.082 | 0.974 |
| 9 | 97.040 | 22.433 | 0.975 |
| 10 | 97.000 | 22.373 | 0.975 |

Epoch 10 is 9700 of 10000 validation images. The printed loss dips and then rises slightly from epoch 6 to epoch 7 while accuracy stays flat. Epoch 9 is a hair above epoch 10 in accuracy. The program does not evaluate the separate test file.


## Conclusion

The training call is one subroutine, `train`, instead of an open `forward` / `backward` / `update` loop. The accuracy function is ordinary Fortran around `maxloc` and `predict`. The loss column needs the shape note above before it is compared with a per-image mean squared error.